In [ ]:
import sys
import subprocess
import time
import requests

from bs4 import BeautifulSoup
import pandas as pd

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


In [ ]:
def scrape_flipkart_mobiles(query="mobiles", max_pages=3, delay=2, output_file="flipkart_mobiles.csv"):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36'
    }
    
    all_data = []
    base_url = "https://www.flipkart.com/search"
    
    print(f"🚀 Scraping '{query}' for {max_pages} pages...\n")
    
    for page in range(1, max_pages + 1):
        print(f"📄 Page {page}...")
        
        params = {"q": query, "page": page}
        response = requests.get(base_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print("   ⚠️ Blocked")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Stronger selectors for current Flipkart layout
        cards = soup.find_all("div", class_="_1AtVbE") or soup.find_all("div", {"data-id": True})
        
        print(f"   Found {len(cards)} product cards")
        
        for card in cards:
            try:
                # Product Name - Multiple possible classes
                name_tag = (card.find("div", class_="KzDlHZ") or 
                           card.find("a", class_="IRpwTa") or 
                           card.find("div", class_="rgWa7D"))
                name = name_tag.text.strip() if name_tag else "N/A"
                
                # Price
                price_tag = card.find("div", class_=re.compile("hZ3P6w|_30jeq3|_16Jk6d"))
                price = price_tag.text.strip() if price_tag else "N/A"
                
                # Rating
                rating_tag = card.find("div", class_="_3LWZlK")
                rating = rating_tag.text.strip() if rating_tag else "N/A"
                
                # Full Description
                desc_tag = card.find("div", class_="col col-7-12") or card.find("ul", class_="_1xgFaf")
                full_desc = desc_tag.text.strip() if desc_tag else "N/A"
                
                # Extract Specs
                text = full_desc.lower()
                product = {
                    "product_name": name,
                    "price": price,
                    "rating": rating,
                    "ram_rom": re.search(r'(\d+\s*gb\s*ram.*?\d+\s*gb)', text).group(0) if re.search(r'(\d+\s*gb\s*ram.*?\d+\s*gb)', text) else "N/A",
                    "display_size": re.search(r'(\d+\.?\d*\s*(cm|inch))', text).group(0) if re.search(r'(\d+\.?\d*\s*(cm|inch))', text) else "N/A",
                    "camera": re.search(r'(\d+mp.*?camera)', text).group(0) if re.search(r'(\d+mp.*?camera)', text) else "N/A",
                    "battery": re.search(r'(\d{3,4}\s*mah)', text).group(0) if re.search(r'(\d{3,4}\s*mah)', text) else "N/A",
                    "processor": re.search(r'(dimensity|snapdragon|exynos|helio|mediatek)', text).group(0) if re.search(r'(dimensity|snapdragon|exynos|helio|mediatek)', text) else "N/A",
                    "warranty": re.search(r'(\d+\s*year)', text).group(0) if re.search(r'(\d+\s*year)', text) else "N/A",
                    "full_description": full_desc,
                    "page": page
                }
                all_data.append(product)
            except:
                continue
                
        time.sleep(delay)
    
    df = pd.DataFrame(all_data)
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\n✅ Done! {len(df)} products saved.")
    return df


In [ ]:
# Run the scraper
df = scrape_flipkart_mobiles(query="mobiles", max_pages=2, delay=2)

# Preview
df.head(10)


🚀 Scraping 'mobiles' for 2 pages...

📄 Page 1...
   Found 24 product cards
📄 Page 2...
   Found 24 product cards

✅ Done! 48 products saved.


,product_name,price,rating,ram_rom,display_size,camera,battery,processor,warranty,full_description,page
0,N/A,"₹9,999",N/A,4 gb ram | 64 gb,43.23 cm,50mp + 2mp | 8mp front camera,5000 mah,helio,1 year,"Samsung Galaxy F07 (Green, 64 GB)4.37,691 Rati...",1
1,N/A,"₹12,999",N/A,4 gb ram | 128 gb,17.12 cm,50mp + 2mp | 8mp front camera,6000 mah,dimensity,1 year,"Samsung Galaxy F70e 5G (Limelight Green, 128 G...",1
2,N/A,"₹26,999",N/A,8 gb ram | 256 gb,17.17 cm,50mp + 2mp | 32mp front camera,7200 mah,dimensity,1 year,"vivo T5x 5G (Cyber Green, 256 GB)4.414,313 Rat...",1
3,N/A,"₹14,999",N/A,4 gb ram | 128 gb,17.02 cm,50mp rear camera,6000 mah,N/A,N/A,"HMD Vibe2 5G (Peach Pink, 128 GB)4.62,967 Rati...",1
4,N/A,"₹17,499",N/A,4 gb ram | 128 gb,17.27 cm,13mp rear camera,7000 mah,dimensity,1 year,"realme P4 Lite 5G (Mosaic Blue, 128 GB)4.37,81...",1
5,N/A,"₹10,499",N/A,4 gb ram | 64 gb,17.13 cm,50mp rear camera,6000 mah,N/A,1 year,"Ai+ Pulse 2 (Purple, 64 GB)4.34,156 Ratings & ...",1
6,N/A,"₹17,999",N/A,6 gb ram | 128 gb,17.02 cm,50mp + 8mp + 2mp | 13mp front camera,5000 mah,exynos,1 year,"Samsung Galaxy F36 5G (Black, 128 GB)4.311,609...",1
7,N/A,"₹12,000",N/A,4 gb ram | 128 gb,17.02 cm,50mp rear camera,5000 mah,N/A,1 year,"Samsung M06 5G (Sage Green, 128 GB)4.13,325 Ra...",1
8,N/A,"₹14,999",N/A,4 gb ram | 128 gb,17.02 cm,50mp rear camera,6000 mah,N/A,N/A,"HMD Vibe2 5G (Nordic Blue, 128 GB)4.62,967 Rat...",1
9,N/A,"₹22,963",N/A,6 gb ram | 128 gb,17.02 cm,50mp rear camera,7200 mah,N/A,N/A,"IQOO Z11x 5G (Prismatic Green, 128 GB)4.3697 R...",1


In [ ]:
# Show summary
print(f"Total Products: {len(df)}")
print("Columns:", df.columns.tolist())

# Clean and convert rating to numeric
df['rating_clean'] = pd.to_numeric(df['rating'], errors='coerce')

print("\nTop 5 Highest Rated Phones:")
top_rated = df.nlargest(5, 'rating_clean')[['product_name', 'price', 'rating', 'rating_clean']]
print(top_rated)

# Optional: Save cleaned version
df.to_csv("flipkart_mobiles_clean.csv", index=False)
print("\n✅ Cleaned file saved as 'flipkart_mobiles_clean.csv'")


Total Products: 48
Columns: ['product_name', 'price', 'rating', 'ram_rom', 'display_size', 'camera', 'battery', 'processor', 'warranty', 'full_description', 'page', 'price_clean', 'rating_clean']

Top 5 Highest Rated Phones:
  product_name    price rating  rating_clean
0          N/A   ₹9,999    N/A           NaN
1          N/A  ₹12,999    N/A           NaN
2          N/A  ₹26,999    N/A           NaN
3          N/A  ₹14,999    N/A           NaN
4          N/A  ₹17,499    N/A           NaN

✅ Cleaned file saved as 'flipkart_mobiles_clean.csv'


In [ ]:
# === Final Clean Analysis ===

print(f"Total Products Scraped: {len(df)}")
print("\nColumns:", df.columns.tolist())

# === Clean Price ===
df['price_clean'] = df['price'].str.replace('₹', '', regex=False)\
                               .str.replace(',', '', regex=False)\
                               .str.strip()\
                               .astype(float, errors='ignore')

# === Clean Rating ===
df['rating_clean'] = pd.to_numeric(df['rating'], errors='coerce')

# Remove rows with missing price or rating if needed
# df = df.dropna(subset=['price_clean', 'rating_clean'])

print("\n=== Top 5 Highest Rated ===")
top_rated = df.nlargest(5, 'rating_clean')[['product_name', 'price', 'rating_clean', 'battery', 'processor']]
print(top_rated)

print("\n=== Top 5 Most Expensive ===")
top_exp = df.nlargest(5, 'price_clean')[['product_name', 'price_clean', 'rating_clean', 'ram_rom']]
print(top_exp)

print("\n=== Summary Statistics ===")
print(f"Average Rating     : {df['rating_clean'].mean():.2f}")
print(f"Median Rating      : {df['rating_clean'].median():.2f}")
print(f"Highest Rating     : {df['rating_clean'].max()}")
print(f"Products with Rating: {df['rating_clean'].notna().sum()} out of {len(df)}")

# Save cleaned version
df.to_csv("flipkart_mobiles_cleaned.csv", index=False)
print("\n✅ Cleaned data saved as 'flipkart_mobiles_cleaned.csv'")


Total Products Scraped: 48

Columns: ['product_name', 'price', 'rating', 'ram_rom', 'display_size', 'camera', 'battery', 'processor', 'warranty', 'full_description', 'page']

=== Top 5 Highest Rated ===
  product_name    price  rating_clean   battery  processor
0          N/A   ₹9,999           NaN  5000 mah      helio
1          N/A  ₹12,999           NaN  6000 mah  dimensity
2          N/A  ₹26,999           NaN  7200 mah  dimensity
3          N/A  ₹14,999           NaN  6000 mah        N/A
4          N/A  ₹17,499           NaN  7000 mah  dimensity

=== Top 5 Most Expensive ===
   product_name  price_clean  rating_clean             ram_rom
19          N/A      33999.0           NaN  12 gb ram | 256 gb
43          N/A      33999.0           NaN  12 gb ram | 512 gb
2           N/A      26999.0           NaN   8 gb ram | 256 gb
36          N/A      25999.0           NaN   8 gb ram | 256 gb
44          N/A      25999.0           NaN   8 gb ram | 256 gb

=== Summary Statistics ===
Average